# LULC Classification: Uncertainty Quantification Pipeline
**Project:** Multispectral Image Analysis & Uncertainty Quantification  
**Author:** Danesh Selwal  
**Date:** 2026-05-02

---
## Executive Summary
This notebook evaluates post-hoc uncertainty over pretrained LULC classifiers and exports quantitative and spatial outputs for analysis.

**Objective:**
Apply uncertainty estimation methods, compare model behavior, and export reproducible artifacts to esults/ for reporting.

---
## 1. Environment Setup & Configuration
Import dependencies, configure runtime paths, and initialize reproducibility settings before running evaluation.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os
import sys

MODULE_NAME = 'multi_cp'
REPO_ROOT = Path("/content/drive/MyDrive/Dias_Uncertainty_Quantification")
MODULE_DIR = REPO_ROOT / MODULE_NAME
RESULTS_DIR = MODULE_DIR / 'results'
MODELS_DIR = MODULE_DIR / 'models'

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print(f'Module: {MODULE_NAME}')
print(f'Output Directory: {RESULTS_DIR}')


In [1]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # Suppress TensorFlow C++ INFO/WARNING logs
import os
import sys
import io
import json
import math
import gc
import random
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib import cm
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
from openpyxl import Workbook, load_workbook
from openpyxl.drawing.image import Image as XLImage
from openpyxl.utils.dataframe import dataframe_to_rows
from scipy.spatial import Voronoi, voronoi_plot_2d
from sklearn.model_selection import train_test_split

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import backend as K, layers, activations, optimizers
from tensorflow.keras.layers import Input, Add, Multiply, Reshape, Dense, Activation, BatchNormalization, Flatten, Dropout, concatenate, Lambda
from tensorflow.keras.layers import Conv2D, AveragePooling2D, MaxPooling2D, GlobalAveragePooling2D, GlobalAvgPool2D, DepthwiseConv2D, SeparableConv2D, MaxPool2D, UpSampling2D
from tensorflow.keras.layers import Conv2DTranspose, add, multiply, LayerNormalization
from tensorflow.python.util.tf_export import keras_export
from tensorflow.python.ops import array_ops
from tensorflow.python.keras.utils import control_flow_util
from tensorflow.keras.models import load_model, Model

sns.set(style='whitegrid', context='talk')
np.random.seed(42)
random.seed(42)
tf.random.set_seed(42)

PROJECT_ROOT = REPO_ROOT
DATA_DIR = REPO_ROOT / "data"
MODEL_DIR = MODELS_DIR
OUTPUT_DIR = RESULTS_DIR
WORKBOOK_PATH = RESULTS_DIR / "multicp_results.xlsx"
MODEL_REGISTRY_PATH = MODEL_DIR / 'model_registry_multihead.json'
DATA_FILE = DATA_DIR / "DIAS.mat"
LABEL_FILE = DATA_DIR / "DIAS_ref.mat"
H, W, B = 330, 307, 6
P_S = 9
BATCH_SIZE = 128
TRAIN_PERCENT = 0.75
ALPHA = 0.05
SCORING_METHODS = ['RAPS', 'SAPS']
K_HEADS = 7
NUM_CLASSES = 7
UNCERTAIN_FRACTION = 0.10
repo_path = PROJECT_ROOT / 'Multi-CP'
if not repo_path.exists():
    subprocess.run(['git', 'clone', 'https://github.com/yamtawa/Multi-CP.git', str(repo_path)], check=True)
if str(repo_path) not in sys.path:
    sys.path.append(str(repo_path))
from utils import compute_scores
print('Project root:', PROJECT_ROOT)
print('Workbook path:', WORKBOOK_PATH)
print('Multi-CP repo:', repo_path)





Mounted at /content/drive
Project root: /content/drive/My Drive/m_p
Workbook path: /content/drive/My Drive/m_p/uncertainty_results/multicp_results.xlsx
Multi-CP repo: /content/drive/My Drive/m_p/Multi-CP


## 2. Data Ingestion & Preprocessing
Load multispectral inputs and reference labels, apply normalization, and prepare tensors for multi-head training/evaluation.


In [2]:
x = pd.read_csv(DATA_FILE).to_numpy(dtype=np.float32).reshape(H, W, B)
y = pd.read_csv(LABEL_FILE).to_numpy(dtype=np.int32).reshape(H, W)
for b in range(B):
    band = x[:, :, b]
    denom = max(float(np.max(band) - np.min(band)), 1e-8)
    x[:, :, b] = (band - np.min(band)) / denom
pad_width = int((P_S - 1) / 2)
padded_x = np.pad(x, [(pad_width, pad_width), (pad_width, pad_width), (0, 0)], 'edge')
X, Y = [], []
for a in range(H):
    for b in range(W):
        if y[a][b] != 0:
            X.append(padded_x[a:a + P_S, b:b + P_S, :])
            Y.append(y[a][b] - 1)
X = np.array(X, dtype=np.float32)
Y = np.array(Y, dtype=np.int32)
x_train, x_test, y_train, y_test = train_test_split(X, Y, train_size=TRAIN_PERCENT, stratify=Y, random_state=10)
x_cal, x_test_new, y_cal, y_test_new = train_test_split(x_test, y_test, test_size=0.5, stratify=y_test, random_state=42)
print('x_cal:', x_cal.shape, 'x_test_new:', x_test_new.shape)



x_cal: (2155, 9, 9, 6) x_test_new: (2155, 9, 9, 6)


In [3]:
def plot_accuracy_loss_curve(history, use_pearson_corr = None, folder_path = None):
    train_loss = history.history['loss']
    val_loss = history.history['val_loss']
    train_accuracy = history.history['accuracy']
    val_accuracy = history.history['val_accuracy']

    fig = plt.figure(figsize = (24,8))
    ax = plt.subplot(1,1,1)
    ax2 = ax.twinx()
    ax.plot(train_accuracy, color='blue', marker='o', linewidth=1.5, markersize = 2,  label = 'train_accuracy')
    ax.plot(val_accuracy, color='green', marker='o', linewidth=1.5, markersize = 2, label = 'val_accuracy')
    ax.grid()
    plt.xlabel('no. of epoches')
    plt.ylabel('accuracy')
    ax.legend()
    ax2.plot(train_loss, color = 'black', marker='o', linewidth=1.5, markersize = 2, label = 'train_loss')
    ax2.plot(val_loss, color = 'red', marker='o', linewidth=1.5, markersize = 2, label = 'val_loss')
    ax2.grid()
    plt.xlabel('no. of epoches')
    plt.ylabel('loss')
    plt.title('accuracy and loss plot for model performance')
    ax2.legend()
    plt.show()
    if folder_path:
        results_dir = os.path.join(folder_path, "Results")
        os.makedirs(results_dir, exist_ok=True) # Create the directory if it doesn't exist
        if use_pearson_corr:
            path = os.path.join(results_dir, "Pearson_Corr " + str(train_percent) + "% ps_"+ str(P_S) + " accuracy_loss.png")
        else:
            path = os.path.join(results_dir, str(train_percent) + "% ps_"+ str(P_S) + " accuracy_loss.png")
        fig.savefig(path)

def performance_meausures(y_test, y_pred, tt, *parameters_summary, folder_path = None):
    Total_params, Trainable_params, Non_trainable_params = parameters_summary
    accuracy = accuracy_score(y_test, y_pred)
    kappa=cohen_kappa_score(y_test, y_pred)
    cm = confusion_matrix(y_test, y_pred).astype('int32')
    cr = classification_report(y_test, y_pred, output_dict=True)
    df_cr = pd.DataFrame(cr).T
    df_score = pd.DataFrame({'accuracy score: ' : [accuracy], 'Cohen_Kappa score: ' : [kappa], "Training Time: " : [tt]}).T
    df_summary = pd.DataFrame({'Total_params: ': [Total_params], 'Trainable_params: ' : [Trainable_params], 'Non_trainable_params: ':[Non_trainable_params]}).T

    spec = gridspec.GridSpec(ncols = 2, nrows = 2, width_ratios=[1,3], wspace = 0.5, hspace = 0.5, height_ratios=[7,1])

    fig = plt.figure(figsize = (24,10))

    ax1 = fig.add_subplot(spec[0])
    ax1.set_title('classification report')
    sns.heatmap(df_cr, cmap = 'Blues', cbar = False, annot = True, fmt=' .5g', ax = ax1)

    ax2 = fig.add_subplot(spec[1])
    ax2.set_title('confusion matrix')
    ax2.set_xlabel('predicted class')
    ax2.set_ylabel('actual class')
    sns.heatmap(cm, cmap = 'Blues', cbar = False, annot = True, fmt=' .5g', ax = ax2)

    ax3 = fig.add_subplot(spec[2])
    sns.heatmap(df_score, cmap = 'Blues', cbar = False, annot = True, fmt=' .5g', ax = ax3)
    ax3.set_xticks([])

    ax4 = fig.add_subplot(spec[3])
    sns.heatmap(df_summary, cmap = "Blues", cbar = False, annot = True, fmt=' .10g', ax = ax4)
    ax4.set_xticks([])

    if folder_path:
        path = folder_path + "Results/" + str(train_percent) + "% ps_" + str(P_S) +" Performance Measure.png"
        fig.savefig(path)

class Pearson_correlation_masked(layers.Layer):
    def __init__(self, P_S = 9, **kwargs):
        super(Pearson_correlation_masked, self).__init__(**kwargs)
        self.P_S = P_S

    def call(self, inputs):
        self.loc = (self.P_S)//2
        self.inputs = inputs
        self.channels = self.inputs.shape[-1]
        self.x_mean = tf.repeat(tf.math.reduce_mean(self.inputs, axis = -1, keepdims=True), repeats = self.channels, axis = -1)

        self.y = tf.repeat(tf.repeat(self.inputs[:,self.loc:self.loc+1, self.loc:self.loc+1, :], repeats = self.P_S, axis = -2), repeats = self.P_S, axis = -3)

        self.y_mean = tf.repeat(tf.math.reduce_mean(self.y, axis = -1, keepdims = True), repeats = self.channels, axis = -1)

        self.a = tf.math.subtract(self.inputs, self.x_mean)
        self.b = tf.math.subtract(self.y, self.y_mean)
        self.ab = tf.math.multiply(self.a,self.b)
        self.num = tf.math.reduce_sum(self.ab, axis = -1, keepdims = True)

        self.a_new = tf.math.reduce_sum(tf.math.multiply(self.a, self.a), axis = -1, keepdims = True)
        self.b_new = tf.math.reduce_sum(tf.math.multiply(self.b, self.b), axis = -1, keepdims = True)
        self.deno = tf.math.sqrt(tf.math.multiply(self.a_new, self.b_new))

        self.corr = tf.math.divide(self.num, self.deno)

        self.thresh = tf.math.reduce_mean(self.corr)
        self.mask = self.corr > self.thresh
        self.mask = tf.cast(self.mask, self.corr.dtype)

        self.masked_corr = tf.math.multiply(self.mask, self.corr)

        self.attention_weights = tf.repeat(self.masked_corr, repeats = self.channels, axis = -1)
        return multiply([self.inputs, self.attention_weights])

    def get_config(self, **kwargs):
        config = super(Pearson_correlation_masked, self).get_config()
        config.update({
            "P_S": self.P_S,
        })
        return config

@keras_export('keras.layers.Dropout')
class Dropout_Train(layers.Layer):
    def __init__(self, rate, shift = 1, noise_shape=None, seed=None, **kwargs):
        super(Dropout_Train, self).__init__(**kwargs)

        if isinstance(rate, (int, float)) and not 0 <= rate <= 1:
            raise ValueError(f"Invalid value {rate} received for `rate`, expected a value between 0 and 1.")
        if type(shift) != int:
            raise TypeError(f"Invalid dtype {type(shift)} found for `shift`. It must be an integer")
        if shift*rate > 1.0:
            raise ValueError(f"Invalid value {shift} received for `shift`, expected an integer value less than or equal to {int(1/rate)}")
        self.rate = rate
        self.shift = shift
        self.noise_shape = noise_shape
        self.seed = seed
        self.supports_masking = True

    def _get_noise_shape(self, inputs):
        if self.noise_shape is None:
            return None

        concrete_inputs_shape = array_ops.shape(inputs)
        noise_shape = []
        for i, value in enumerate(self.noise_shape):
            noise_shape.append(concrete_inputs_shape[i] if value is None else value)
        return tf.convert_to_tensor(noise_shape)

    def call(self, inputs, training=None):
        if self.rate == 0:
            return tf.identity(inputs)

        if training is None:
            training = K.learning_phase()

        def dropped_inputs():
            input_shape = inputs.shape
            range_0 = int(self.rate*(self.shift-1)*input_shape[-1])
            if self.shift*self.rate < 1.0:
                range_1 = int(self.rate*(self.shift)*input_shape[-1])
            else:
                range_1 = None
            input_shape = inputs.shape
            multiplier = np.ones(input_shape[-1])
            multiplier[range_0:range_1] = 0.0
            multiplier = tf.constant(multiplier)
            return Multiply()([inputs, multiplier])

        output = control_flow_util.smart_cond(training, dropped_inputs, lambda: array_ops.identity(inputs))
        return output

    def compute_output_shape(self, input_shape):
        return input_shape

    def get_config(self):
        config = super(Dropout_Train, self).get_config()
        config.update({
            "rate": self.rate,
            "shift": self.shift,
            "noise_shape": self.noise_shape,
            "seed": self.seed,
            "supports_masking": self.supports_masking
        })
        return config

def modified_model(model, layer_name, rate, new_layer, shift, **kwargs):    # layer_name = "TRAIN_DROPOUT"
    name = kwargs["name"] if kwargs else None
    x = model.layers[0].output
    modification = False
    z = 0
    for lyr in model.layers[1:]:
        if (layer_name in lyr.name or layer_name in lyr.name.upper()) and (type(shift) != str):
            x = new_layer(rate = rate, shift = shift, name = layer_name + "_" + str(shift) + "_" + str(z))(x)
            modification = True
            z += 1
        elif (layer_name in lyr.name or layer_name in lyr.name.upper()) and (type(shift) == str):
            x = new_layer(rate = rate, name = layer_name + "_" + str(shift) + "_" + str(z))(x)
            z += 1
            modification = True
        else:
            x = lyr(x)
    if not modification:
        print("___________________________________Model has not been modified___________________________________")
    return Model(inputs = model.layers[0].input, outputs = x, name = name)

class Custom_callbacks(tf.keras.callbacks.Callback):
    def __init__(self, filepath, epochs, rate, new_layer = Dropout_Train, layer_name = "DROPOUT", accuracy_score = 0.99, min_epochs = 50):
        super(Custom_callbacks, self).__init__()
        self.filepath = filepath
        self.epochs = epochs
        self.new_layer = new_layer
        self.rate = rate
        self.best = 0.0
        self.epoch_num = 1
        self.layer_name = layer_name
        self.min_epochs = min_epochs                    # minimum number of epochs that model should be trained in each shift
        self.accuracy_score = accuracy_score if accuracy_score <= 1.0 else accuracy_score/100.0

    def on_train_begin(self, logs=None):
        print(self.epochs)
        keys = list(logs.keys())
        self.shift = 1
        self.epoch_completed = 0
        print(f"Model will be trained in {int(1/self.rate)} shifts")
        print("Starting training with 1st shift \n")
        self.model = modified_model(self.model, self.layer_name, self.rate, self.new_layer, self.shift)

    def on_train_end(self, logs=None):
        keys = list(logs.keys())
        if self.shift <= int(1/self.rate):
            raise NotImplementedError(f"model has not trained fully in the available no. of epochs \n only {self.shift-1} shifts completed out of {int(1/self.rate)}")
        print("Model training completition ", "███████████"*self.shift, (self.rate*(self.shift-1))*100, "%")
        print(f"Model has been fully trained in {int(1/self.rate)} shifts")
        self.model.set_weights(self.best_weights)
        print(f"\nSaving best model to {self.filepath}")
        self.model.save(self.filepath)

    def on_epoch_end(self, epoch, logs=None):
        keys = list(logs.keys())
        self.epoch_completed += 1
        self.epoch_num += 1

        if (logs["val_accuracy"] >= self.accuracy_score) and (self.epoch_completed >= self.min_epochs) and (self.shift < int(1/self.rate)):
            print("\nTargeted accuracy has been achieved")
            print("Model training completition ", "███████████"*(self.shift), (self.rate*self.shift)*100, "%")
            self.shift += 1
            Suffixes = "nd" if (self.shift == 2) else "th"
            print(f"Modifying the model for {self.shift}{str(Suffixes)} shift")
            self.model = modified_model(self.model, self.layer_name, self.rate, self.new_layer, self.shift)
            self.epoch_completed = 0

        elif (logs["val_accuracy"] >= self.accuracy_score) and (self.epoch_completed >= self.min_epochs) and (self.shift == int(1/self.rate)):
            print("\nModel training completition ", "███████████"*(self.shift), (self.rate*self.shift)*100, "%")
            print("All shifting has been completed\n")
            print("██████████████████████===============> Now redefining the model to standard model <===============██████████████████████")
            self.model = modified_model(self.model, self.layer_name, self.rate, self.new_layer, "Final", name = "AlexNet")
            self.shift += 1
            self.epoch_completed = 0

        else:
            print(", need more training")
            if self.shift >= int(1/self.rate):
                current = logs.get("val_accuracy")
                if not np.less(current, self.best) and (self.epoch_num >= self.epochs-10):
                    print(f"val_accuracy improved from {self.best:.4f} to {current:.4f}")
                    self.best = current
                    self.best_weights = self.model.get_weights()

    def get_config(self):
        config = super(Custom_callbacks, self).get_config()
        config.update({
            "filepath": self.filepath,
            "epochs": self.epochs,
            "new_layer": self.new_layer,
            "rate": self.rate,
            "best": self.best,
            "epoch_num": self.epoch_num,
            "layer_name": self.layer_name,
            "min_epochs": self.min_epochs,
            "accuracy_score": self.accuracy_score,
        })
        return config


In [4]:
def AlexNet(input_shape, num_classes=13, use_pearson_corr=False, dropout_rate=0.5):
    K_HEADS = 7  # Number of separate output heads

    x_input = Input(input_shape)
    if use_pearson_corr:
        X = Pearson_correlation_masked(P_S)(x_input)
    else:
        X = x_input

    # Convolutional backbone
    X = Conv2D(filters=96, kernel_size=(3,3), activation='relu', strides=(1,1), padding='same')(X)
    X = Conv2D(filters=256, kernel_size=(3,3), activation='relu', strides=(1,1), padding='same')(X)
    X = Conv2D(filters=384, kernel_size=(3,3), activation='relu', strides=(1,1), padding='same')(X)
    X = Conv2D(filters=384, kernel_size=(3,3), activation='relu', strides=(1,1), padding='same')(X)
    X = Conv2D(filters=256, kernel_size=(3,3), activation='relu', strides=(1,1), padding='same')(X)
    X = MaxPooling2D(pool_size=(2,2), strides=(2,2), padding='same')(X)

    # Flatten and dense layers
    X = Flatten()(X)
    X = Dense(4096, activation='relu')(X)
    X = Dropout(dropout_rate, name="TRAIN_DROPOUT_1")(X)
    X = Dense(1024, activation='relu')(X)
    X = Dropout(dropout_rate, name="TRAIN_DROPOUT_2")(X)
    X = Dense(256, activation='relu')(X)
    X = Dropout(dropout_rate, name="TRAIN_DROPOUT_3")(X)
    X = Dense(32, activation='relu')(X)

    # --- START OF MULTI-HEAD MODIFICATION ---
    output_heads = []
    for i in range(K_HEADS):
        head_name = f'head_{i+1}'
        head_output = Dense(num_classes, activation='softmax', dtype='float32', name=head_name)(X)
        output_heads.append(head_output)

    model = Model(inputs=x_input, outputs=output_heads, name="MultiHead_AlexNet")
    # --- END OF MULTI-HEAD MODIFICATION ---

    return model



In [5]:
class GF_MLP(layers.Layer):
    def __init__(self, in_features, out_features, drop = 0.0, **kwargs):
        super(GF_MLP, self).__init__(**kwargs)
        self.in_features = in_features
        self.out_features = out_features
        self.drop = drop
        self.mlp_1 = Dense(in_features, activation = activations.gelu, use_bias = False)
        self.drop_1 = Dropout(drop)
        self.mlp_2 = Dense(out_features, activation = activations.gelu, use_bias = False)
        self.drop_2 = Dropout(drop)

    def call(self, x):
        x = self.drop_1(self.mlp_1(x))
        x = self.drop_2(self.mlp_2(x))
        return x

    def get_config(self, **kwargs):
        config = super(GF_MLP, self).get_config()
        config.update({
            "in_features": self.in_features,
            "out_features": self.out_features,
            "drop": self.drop,
        })
        return config

class GF_DropPath(layers.Layer):
    def __init__(self, drop_prob = 0.0, training = False, **kwargs):
        super(GF_DropPath, self).__init__(**kwargs)
        self.drop_prob = drop_prob
        self.training = training
    def call(self, x, **kwargs):
        if self.drop_prob == 0. or not self.training:
            return x
        keep_prob = 1 -  self.drop_prob
        shape = (x.shape[0],) + (1,) * (x.ndim - 1)
        random_tensor = keep_prob + tf.random.uniform(shape, dtype=x.dtype)
        random_tensor.floor_()  # binarize
        output = tf.divide(x, keep_prob) * random_tensor
        return output
    def get_config(self, **kwargs):
        config = super(GF_DropPath, self).get_config()
        config.update({
            "drop_prob": self.drop_prob,
            "training": self.training,
        })
        return config

class GF_Expand_Dims(layers.Layer):
    def __init__(self, ndim, **kwargs):
        super(GF_Expand_Dims, self).__init__(**kwargs)
        self.ndim = ndim

    def call(self, x):
        x = tf.expand_dims(x, axis = self.ndim)
        return x

    def config(self, **kwargs):
        config = super(GF_Expand_Dims, self).config()
        config.update({
            "ndim" : self.ndim,
        })
        return config

class GF_Patches(layers.Layer):
    def __init__(self, patch_size = 3, hidden_dim = 256, patch_method='extract', **kwargs):
        super(GF_Patches, self).__init__(**kwargs)
        self.patch_size = patch_size
        self.hidden_dim = hidden_dim
        self.patch_method = patch_method.lower()

    def call(self, images):
        if self.patch_method == "conv":
            x = Conv2D(self.hidden_dim, self.patch_size, self.patch_size)(images)              # shape(B, 3, 3, 256)
            patches = Reshape([-1, x.shape[-1]])(x)                                   # shape(B, 9, 256)
            return patches
        else:
            batch_size = tf.shape(images)[0]
            patches = tf.image.extract_patches(images=images, sizes=[1, self.patch_size, self.patch_size, 1], strides=[1, self.patch_size, self.patch_size, 1],
                                            rates=[1, 1, 1, 1], padding="VALID",)
            patch_dims = patches.shape[-1]
            patches = tf.reshape(patches, [batch_size, -1, patch_dims])
            return patches
    def get_config(self):
        config = super(GF_Patches, self).get_config()
        config.update({
            "patch_size": self.patch_size,
            "hidden_dim": self.hidden_dim,
            "patch_method": self.patch_method
        })
        return config

class GF_PatchEncoder(layers.Layer):
    def __init__(self, num_patches, projection_dim, **kwargs):
        super(GF_PatchEncoder, self).__init__(**kwargs)
        self.num_patches = num_patches
        self.projection_dim = projection_dim
        self.projection = layers.Dense(units=projection_dim)
        self.position_embedding = layers.Embedding(input_dim=num_patches, output_dim=projection_dim)

    def call(self, patch,**kwargs):
        positions = tf.range(start=0, limit=self.num_patches, delta=1)
        encoded = self.projection(patch) + self.position_embedding(positions)
        return encoded

    def get_config(self, **kwargs):
        config = super(GF_PatchEncoder, self).get_config()
        config.update({
            "num_patches": self.num_patches,
            "projection_dim": self.projection_dim,
        })
        return config

class GF_GlobalFilter(layers.Layer):
    def __init__(self, patch_size, dim, **kwargs):
        super(GF_GlobalFilter, self).__init__(**kwargs)
        self.patch_size = patch_size
        self.dim = dim

    def build(self, input_shape):
        w_init = tf.random_uniform_initializer()
        self.complex_weight = self.add_weight(
            name="complex_weight",
            shape=(self.patch_size, self.patch_size, input_shape[-1] // 2 + 1, 2),
            initializer=w_init,
            trainable=True
        )
        super().build(input_shape)

    def call(self, x, **kwargs):
        B, N, C = x.shape
        a = b = int(math.sqrt(N))
        x = tf.reshape(x, [-1, a, b, C])
        x = tf.signal.rfft2d(x)
        weight = tf.dtypes.complex(self.complex_weight[:,:,:,0], self.complex_weight[:,:,:,-1])
        x = x * weight
        x = tf.signal.irfft2d(x)
        x = tf.reshape(x, [-1, N, C])
        return x

    def config(self, **kwargs):
        config = super(GF_GlobalFilter, self).config()
        config.update({
            "patch_size" : self.patch_size,
            "dim" : self.patch_size,
        })
        return config

class GF_Block(tf.keras.layers.Layer):
    def __init__(self, patch_size=3, dim=512, mlp_ratio=4.0, drop=0.0, drop_path=0.0, **kwargs):
        super(GF_Block, self).__init__(**kwargs)
        self.patch_size = patch_size
        self.dim = dim
        self.mlp_ratio = mlp_ratio
        self.drop = drop
        self.drop_path_rate = drop_path

        # Define sublayers
        self.norm1 = tf.keras.layers.LayerNormalization(axis=-1)
        self.filter = GF_GlobalFilter(patch_size, dim)  # Make sure GlobalFilter uses add_weight inside its build()
        self.drop_path = GF_DropPath(drop_path)
        self.norm2 = tf.keras.layers.LayerNormalization(axis=-1)
        mlp_hidden_dim = int(dim * mlp_ratio)
        self.mlp = GF_MLP(in_features=mlp_hidden_dim, out_features=dim, drop=drop)

    def call(self, x):
        # Transformer-like residual block
        x = x + self.drop_path(self.mlp(self.norm2(self.filter(self.norm1(x)))))
        return x

    def get_config(self):
        config = super(GF_Block, self).get_config()
        config.update({
            "patch_size": self.patch_size,
            "dim": self.dim,
            "mlp_ratio": self.mlp_ratio,
            "drop": self.drop,
            "drop_path": self.drop_path_rate,
        })
        return config

def GFNet(input_shape=(P_S, P_S, B),
          use_pearson_corr=False,
          patch_size=3,
          num_classes=16,
          hidden_dim=512,
          GlobalFilter_layers=12,
          mlp_ratio=4,
          num_patches=9,
          dropout_rate=0.0,
          drop_path_rate=0.0):

    K_HEADS = 7  # Number of separate output heads

    # --- Input & optionally apply Pearson correlation ---
    x_input = Input(shape=input_shape)
    if use_pearson_corr:
        x = Pearson_correlation_masked(P_S)(x_input)
    else:
        x = x_input

    # --- Patch extraction and encoding ---
    x = GF_Patches(patch_size)(x)                     # shape e.g. (num_patches, patch_dim)
    x = GF_PatchEncoder(num_patches, hidden_dim)(x)   # encodes patches
    x = Dropout(dropout_rate, name="TRAIN_DROPOUT_1")(x)

    # --- Global Filter layers ---
    for _ in range(GlobalFilter_layers):
        x = GF_Block(
            patch_size=patch_size,
            dim=hidden_dim,
            mlp_ratio=mlp_ratio,
            drop=dropout_rate,
            drop_path=drop_path_rate
        )(x)

    # --- Pooling and normalization ---
    x = Dropout(dropout_rate, name="TRAIN_DROPOUT_2")(x)
    x = LayerNormalization()(x)
    x = GF_Expand_Dims(ndim=2)(x)
    x = GlobalAveragePooling2D()(x)
    x = Flatten()(x)
    x = Dropout(dropout_rate, name="TRAIN_DROPOUT_3")(x)

    # --- START OF MULTI-HEAD MODIFICATION ---
    output_heads = []
    for i in range(K_HEADS):
        head_name = f"head_{i+1}"
        head_output = Dense(num_classes, activation="softmax", dtype="float32", name=head_name)(x)
        output_heads.append(head_output)

    model = keras.Model(inputs=x_input, outputs=output_heads, name="MultiHead_GFNet")
    # --- END OF MULTI-HEAD MODIFICATION ---

    return model


In [6]:
class ViT_SpatialAttention(layers.Layer):
    def __init__(self, k_size=3, **kwargs):
        super().__init__(**kwargs)
        self.k_size = k_size
        self.norm = layers.BatchNormalization()
        self.conv1 = layers.Conv2D(1, kernel_size=(k_size, k_size), padding="same")
        self.conv2 = layers.Conv2D(1, kernel_size=(k_size, k_size), padding="same")
        self.conv3 = layers.Conv2D(1, kernel_size=(k_size, k_size), padding="same")
        self.conv4 = layers.Conv2D(1, kernel_size=(k_size, k_size), padding="same")
        self.act_relu = layers.Activation("relu")
        self.act_sigmoid = layers.Activation("sigmoid")

    def call(self, inputs):
        x = self.conv1(inputs)
        x = self.norm(x)
        x = self.conv2(x)
        x = self.act_relu(x)
        x = self.conv3(x)
        x = self.act_relu(x)
        x = self.conv4(x)
        return self.act_sigmoid(x)

    def get_config(self):
        config = super().get_config()
        config.update({"k_size": self.k_size})
        return config

class ViT_SpatialAttention1(layers.Layer):
    def __init__(self, input_shape, **kwargs):
        super().__init__(**kwargs)
        self.input_shape_val = input_shape
        self.filters = input_shape[-1]
        self.k_size = input_shape[1]

        self.norm = layers.BatchNormalization()
        self.conv1 = layers.Conv2D(self.filters, kernel_size=3, padding="same", kernel_initializer="he_normal")
        self.conv2 = layers.Conv2D(self.filters, kernel_size=3, strides=2, padding="same")
        self.conv3 = layers.Conv2D(self.filters, kernel_size=3, strides=2, padding="same")

        self.convt1 = layers.Conv2DTranspose(self.filters, kernel_size=3, strides=2, padding="same")
        self.convt2 = layers.Conv2DTranspose(self.filters, kernel_size=3, strides=2, padding="same")

        self.relu = layers.ReLU()
        self.sigmoid = layers.Activation("sigmoid")

    def call(self, inputs):
        x = self.conv1(inputs)
        x = self.norm(x)
        x = self.relu(x)

        x = self.conv2(x)
        x = self.relu(x)

        x = self.conv3(x)
        x = self.relu(x)

        x = self.convt1(x)
        x = self.relu(x)

        x = self.convt2(x)
        x = self.relu(x)

        # shape correction if needed
        if x.shape[1] != self.input_shape_val[1] or x.shape[2] != self.input_shape_val[2]:
            k_size = x.shape[1] - self.k_size + 1
            x = layers.Conv2D(self.filters, kernel_size=k_size, strides=1, padding="valid")(x)

        return self.sigmoid(x)

    def get_config(self):
        config = super().get_config()
        config.update({
            "input_shape": self.input_shape_val,
            "filters": self.filters,
            "k_size": self.k_size
        })
        return config

# Implement Feed Forward Network / MLP
def MLP(x, hidden_units, dropout_rate):
    """
    Feedforward network used inside Transformer blocks.
    hidden_units: list of layer sizes (e.g. [mlp_dim, embed_dim])
    dropout_rate: dropout between dense layers
    """
    for units in hidden_units:
        # Use GELU activation, which is common in ViTs
        x = layers.Dense(units, activation=tf.keras.activations.gelu)(x)
        x = layers.Dropout(dropout_rate)(x)
    return x


# Implement patch creation as a layer
class ViT_Patches(layers.Layer):
    def __init__(self, patch_size, embed_dim=768, **kwargs):
        super(ViT_Patches, self).__init__(**kwargs)
        self.patch_size = patch_size
        self.embed_dim = embed_dim

    def build(self, input_shape):
        # Flatten each patch across spatial + spectral channels, then project
        _, H, W, C = input_shape
        patch_dim = self.patch_size * self.patch_size * C
        self.projection = layers.Dense(self.embed_dim)

    def call(self, images):
        batch_size = tf.shape(images)[0]

        # Extract non-overlapping patches
        patches = tf.image.extract_patches(
            images=images,
            sizes=[1, self.patch_size, self.patch_size, 1],
            strides=[1, self.patch_size, self.patch_size, 1],
            rates=[1, 1, 1, 1],
            padding="VALID",
        )
        patch_dims = patches.shape[-1]   # = patch_size*patch_size*C
        patches = tf.reshape(patches, [batch_size, -1, patch_dims])

        # Project raw patches to embedding dimension for the transformer
        return self.projection(patches)

    def get_config(self):
        config = super(ViT_Patches, self).get_config()
        config.update({
            "patch_size": self.patch_size,
            "embed_dim": self.embed_dim,
        })
        return config

class ViT_PatchEncoder(layers.Layer):
    def __init__(self, num_patches, projection_dim, **kwargs):
        super(ViT_PatchEncoder, self).__init__(**kwargs)
        self.num_patches = num_patches
        self.projection_dim = projection_dim

        # Linear projection of patches → projection_dim
        self.projection = layers.Dense(units=projection_dim)

        # Position embedding (num_patches + 1 to include CLS token)
        self.position_embedding = layers.Embedding(
            input_dim=num_patches + 1, output_dim=projection_dim
        )

        # Learnable class token
        cls_init = tf.zeros_initializer()
        self.cls_token = self.add_weight(
            name="cls_token",
            shape=(1, 1, projection_dim),
            initializer=cls_init,
            trainable=True,
        )

    def call(self, patch, **kwargs):
        batch_size = tf.shape(patch)[0]

        # Project patches [B, num_patches, projection_dim]
        patch_proj = self.projection(patch)

        # Broadcast cls token for this batch [B, 1, projection_dim]
        cls_tokens = tf.repeat(self.cls_token, repeats=batch_size, axis=0)

        # Concatenate CLS + patch sequence
        x = tf.concat([cls_tokens, patch_proj], axis=1)

        # Add position embeddings
        positions = tf.range(start=0, limit=self.num_patches + 1, delta=1)
        pos_embeddings = self.position_embedding(positions)
        encoded = x + pos_embeddings

        return encoded

    def get_config(self):
        config = super(ViT_PatchEncoder, self).get_config()
        config.update({
            "num_patches": self.num_patches,
            "projection_dim": self.projection_dim,
        })
        return config

class ViT_Weighted_add(layers.Layer):
    def __init__(self, name, **kwargs):
        super(ViT_Weighted_add, self).__init__(**kwargs)
        self.wt_name = name

    def build(self, input_shape):
        # Use add_weight instead of raw tf.Variable
        self.w = self.add_weight(
            name="weighted_add_" + str(self.wt_name),
            shape=(1,),
            initializer=tf.random_normal_initializer(),
            trainable=True
        )

    def call(self, input_1, input_2):
        return input_1 * self.w + input_2 * (1.0 - self.w)

    def get_config(self):
        config = super(ViT_Weighted_add, self).get_config()
        config.update({
            "wt_name": self.wt_name,
        })
        return config

class ViT_TransFormer(layers.Layer):
    def __init__(self, layer_num, num_heads, projection_dim, dropout=0.1, **kwargs):
        super(ViT_TransFormer, self).__init__(**kwargs)
        self.layer_num = layer_num
        self.num_heads = num_heads
        self.projection_dim = projection_dim
        self.dropout = dropout

    def build(self, input_shape):
        self.norm1 = layers.LayerNormalization(epsilon=1e-6, name=f"ln1_{self.layer_num}")
        self.norm2 = layers.LayerNormalization(epsilon=1e-6, name=f"ln2_{self.layer_num}")

        self.add1 = ViT_Weighted_add(f"transformer_1_{self.layer_num}")
        self.add2 = ViT_Weighted_add(f"transformer_2_{self.layer_num}")

        self.mha = layers.MultiHeadAttention(
            num_heads=self.num_heads,
            key_dim=self.projection_dim,
            dropout=self.dropout,
            name=f"mha_{self.layer_num}"
        )

        self.dense1 = layers.Dense(self.projection_dim * 2, activation=tf.keras.activations.gelu)
        self.drop1 = layers.Dropout(self.dropout)

        self.dense2 = layers.Dense(self.projection_dim, activation=tf.keras.activations.gelu)
        self.drop2 = layers.Dropout(self.dropout)

    def call(self, inputs, training=None):
        # Multi-Head Attention block
        x1 = self.norm1(inputs)
        x1 = self.mha(x1, x1, training=training)
        x1 = self.add1(x1, inputs)   # residual with learned weighting

        # Feed Forward block
        x2 = self.norm2(x1)
        x2 = self.dense1(x2)
        x2 = self.drop1(x2, training=training)
        x2 = self.dense2(x2)
        x2 = self.drop2(x2, training=training)

        return self.add2(x2, x1)     # residual with learned weighting

    def get_config(self):
        config = super(ViT_TransFormer, self).get_config()
        config.update({
            "layer_num": self.layer_num,
            "num_heads": self.num_heads,
            "projection_dim": self.projection_dim,
            "dropout": self.dropout
        })
        return config

class ViT_TransFormer_Block(layers.Layer):
    def __init__(self, num_layers, num_heads, projection_dim, dropout=0.1, **kwargs):
        super(ViT_TransFormer_Block, self).__init__(**kwargs)
        self.num_layers = num_layers
        self.num_heads = num_heads
        self.projection_dim = projection_dim
        self.dropout = dropout

    def build(self, input_shape):
        # Build list of TransFormer layers
        self.Blocks = [
            ViT_TransFormer(i, self.num_heads, self.projection_dim, self.dropout)
            for i in range(self.num_layers)
        ]

    def call(self, inputs, training=None):
        block_list = []
        x = inputs
        for i in range(self.num_layers):
            if i <= self.num_layers // 2:
                x = self.Blocks[i](x, training=training)
                block_list.append(x)
            else:
                x = self.Blocks[i](x, training=training)
                # Symmetric skip connection (mirror like U‑Net)
                x = layers.Add()([x, block_list[self.num_layers - i - 1]])
        return x

    def get_config(self):
        config = super(ViT_TransFormer_Block, self).get_config()
        config.update({
            "num_layers": self.num_layers,
            "num_heads": self.num_heads,
            "projection_dim": self.projection_dim,
            "dropout": self.dropout,
        })
        return config

class ViT_Class_Token_Norm(layers.Layer):
    def __init__(self, eps=1e-6, **kwargs):
        super(ViT_Class_Token_Norm, self).__init__(**kwargs)
        self.eps = eps
        self.norm = layers.LayerNormalization(epsilon=self.eps)

    def call(self, inputs):
        # Normalize entire sequence, then extract CLS token
        x = self.norm(inputs)
        cls_token = x[:, 0, :]     # shape (B, D)
        return cls_token

    def get_config(self):
        config = super(ViT_Class_Token_Norm, self).get_config()
        config.update({
            "eps": self.eps
        })
        return config

projection_dim = 256
num_heads = 4
transformer_layers = 12
dropout = 0.1          # dropout inside Transformer blocks

def create_vit_classifier(input_shape=(P_S, P_S, B),
                          num_classes=7,
                          use_pearson_corr=False,
                          dropout_rate=0.25,
                          method="with_gap",
                          k_heads: int = 1):
    """
    Vision Transformer classifier with optional multi-head outputs.
    - k_heads: number of classification heads (each head = independent Dense softmax).
      If k_heads == 1 -> returns a single logits tensor (shape [B, num_classes]).
      If k_heads > 1 -> returns a list of k_heads tensors [ (B,num_classes), ... ].
    """
    inputs = layers.Input(shape=input_shape)

    # Optional Pearson correlation preprocessing
    if use_pearson_corr:
        x0 = Pearson_correlation_masked(P_S)(inputs)
    else:
        x0 = inputs

    # Patch creation
    patches = ViT_Patches(patch_size, embed_dim=projection_dim)(x0)

    # Patch encoding
    encoded_patches = ViT_PatchEncoder(num_patches, projection_dim)(patches)

    # Transformer stack
    encoded_patches = ViT_TransFormer_Block(transformer_layers, num_heads, projection_dim, dropout)(encoded_patches)
    encoded_patches = layers.Dropout(dropout_rate, name="TRAIN_DROPOUT_1")(encoded_patches)

    # Representation strategies
    if method == "with_cls_tkn":
        # normalize + extract CLS token
        representation = ViT_Class_Token_Norm(eps=1e-6)(encoded_patches)
    elif method == "without_gap":
        representation = layers.LayerNormalization(epsilon=1e-6)(encoded_patches)
        representation = layers.Flatten()(representation)
    elif method == "with_gap":
        representation = layers.LayerNormalization(epsilon=1e-6)(encoded_patches)
        representation = tf.reduce_mean(representation, axis=1)  # [B, D]
    else:
        raise ValueError(f"Unknown method: {method}")

    # Classification head (shared backbone -> separate heads)
    x = layers.Dense(512, activation=tf.keras.activations.gelu)(representation)
    x = layers.Dropout(dropout_rate, name="TRAIN_DROPOUT_3")(x)
    x = layers.Dense(256, activation=tf.keras.activations.gelu)(x)
    x = layers.Dense(128, activation=tf.keras.activations.gelu)(x)
    x = layers.Dropout(dropout_rate, name="TRAIN_DROPOUT_5")(x)
    x = layers.Dense(64, activation=tf.keras.activations.gelu)(x)
    features = layers.Dropout(dropout_rate, name="TRAIN_DROPOUT_6")(x)

    # Build k_heads softmax outputs
    output_heads = []
    for i in range(max(1, k_heads)):
        head_name = f"head_{i+1}"
        head_logits = layers.Dense(num_classes, activation="softmax", dtype="float32", name=head_name)(features)
        output_heads.append(head_logits)

    # Return single tensor if only 1 head (backwards compatibility)
    if k_heads == 1:
        model = keras.Model(inputs=inputs, outputs=output_heads[0])
    else:
        model = keras.Model(inputs=inputs, outputs=output_heads)

    return model



In [7]:
CUSTOM_OBJECTS = {
    'Pearson_correlation_masked': Pearson_correlation_masked,
    'Dropout_Train': Dropout_Train,
    'GF_Patches': GF_Patches,
    'GF_PatchEncoder': GF_PatchEncoder,
    'GF_GlobalFilter': GF_GlobalFilter,
    'GF_Block': GF_Block,
    'GF_Expand_Dims': GF_Expand_Dims,
    'GF_MLP': GF_MLP,
    'GF_DropPath': GF_DropPath,
    'ViT_Patches': ViT_Patches,
    'ViT_PatchEncoder': ViT_PatchEncoder,
    'ViT_SpatialAttention': ViT_SpatialAttention,
    'ViT_SpatialAttention1': ViT_SpatialAttention1,
    'ViT_Weighted_add': ViT_Weighted_add,
    'ViT_TransFormer': ViT_TransFormer,
    'ViT_TransFormer_Block': ViT_TransFormer_Block,
    'ViT_Class_Token_Norm': ViT_Class_Token_Norm,
}
MODEL_NAME_MAP = {'AlexNet_CNN_MultiHead': 'AlexNet CNN', 'GFNet_MultiHead': 'GFNet', 'ViT_UNet_MultiHead': 'ViT UNet'}
CLASS_COLORS = ['#0000FF', '#00FF00', '#FF0000', '#00FFFF', '#FF00FF', '#FFFF00', '#A52A2A']
UNCERTAIN_COLOR = '#808080'
CERTAIN_COLOR = '#FFFF00'
UNCERTAIN_MAP_COLOR = '#001F3F'
BINARY_UNCERTAINTY_CMAP = ListedColormap([CERTAIN_COLOR, UNCERTAIN_MAP_COLOR])

def add_bar_labels(ax, fmt='{:.2f}', y_pad=0.01):
    ymax = max(ax.get_ylim()[1], 1e-9)
    for p in ax.patches:
        h = p.get_height()
        if np.isnan(h):
            continue
        ax.text(
            p.get_x() + p.get_width() / 2,
            h + y_pad * ymax,
            fmt.format(h),
            ha='center',
            va='bottom',
            fontsize=9,
        )

def ensure_workbook(path):
    if path.exists():
        return load_workbook(path)
    wb = Workbook(); ws = wb.active; ws.title = 'Summary'; wb.save(path); return wb

def autosize_columns(ws):
    for col in ws.columns:
        values = [len(str(cell.value)) for cell in col if cell.value is not None]
        if values:
            ws.column_dimensions[col[0].column_letter].width = min(max(values) + 2, 40)

def fig_to_buffer(fig):
    buf = io.BytesIO(); fig.savefig(buf, format='png', dpi=220, bbox_inches='tight', facecolor='white'); buf.seek(0); return buf

def add_image(ws, fig, anchor):
    img = XLImage(fig_to_buffer(fig)); img.anchor = anchor; ws.add_image(img)

def write_df(ws, df, start_row=1, start_col=1):
    for r_idx, row in enumerate(dataframe_to_rows(df, index=False, header=True), start=start_row):
        for c_idx, val in enumerate(row, start=start_col):
            ws.cell(row=r_idx, column=c_idx, value=val)

def get_multihead_outputs(model, x_data, batch_size=128):
    outputs = model.predict(x_data, batch_size=batch_size, verbose=0)
    if not isinstance(outputs, list):
        outputs = [outputs]
    return np.stack(outputs, axis=0)

def get_image_multi_head_outputs(model, padded_x, H, W, B, P_S, batch_size=32):
    N = H * W
    patches = np.zeros((N, P_S, P_S, B), dtype=padded_x.dtype)
    idx = 0
    for i in range(H):
        for j in range(W):
            patches[idx] = padded_x[i:i + P_S, j:j + P_S, :]
            idx += 1
    preds = model.predict(patches, batch_size=batch_size, verbose=0)
    return np.stack(preds, axis=0)

def generate_Dcal_Dcells_sets(cal_scores, cal_target, fraction=0.05, seed=42):
    K, N, _ = cal_scores.shape
    rng = np.random.default_rng(seed)
    n_cells = max(1, int(N * fraction))
    idx_cells = rng.choice(N, n_cells, replace=False)
    Dcells_scores = cal_scores[:, idx_cells, cal_target[idx_cells].astype(int)].T
    Dcells_target = cal_target[idx_cells]
    mask = np.ones(N, dtype=bool)
    mask[idx_cells] = False
    Dre_cal_scores = cal_scores[:, mask, :]
    Dre_cal_target = cal_target[mask]
    return Dcells_scores, Dcells_target, Dre_cal_scores, Dre_cal_target

def main_algo(Dcells_scores, Dcells_target, Dre_cal_scores, Dre_cal_target, test_scores, test_target, alpha, config):
    K = Dre_cal_scores.shape[0]
    N_cal = Dre_cal_scores.shape[1]
    cal_true = Dre_cal_scores[np.arange(K)[:, None], np.arange(N_cal), Dre_cal_target]
    q = np.quantile(cal_true, 1 - alpha, axis=1)
    prediction_sets = test_scores <= q[:, None, None]
    valid = (test_target >= 0) & (test_target < prediction_sets.shape[2])
    covered = np.all(prediction_sets[np.arange(K)[:, None], np.arange(np.sum(valid)), test_target[valid]], axis=0)
    return covered.mean(), prediction_sets.sum(axis=2).mean(), prediction_sets

def compute_head_sweep(cal_output, test_output, cal_target, test_target, scoring_method):
    config = {'ALPHA': ALPHA, 'SCORING_METHOD': scoring_method}
    cal_scores = np.round(compute_scores(cal_output, config), 4)
    test_scores = np.round(compute_scores(test_output, config), 4)
    rows = []
    last_bundle = None
    for nH in range(1, cal_output.shape[0] + 1):
        Dc, Dt, Rc, Rt = generate_Dcal_Dcells_sets(cal_scores[:nH], cal_target)
        cov, msz, pred_sets = main_algo(Dc, Dt, Rc, Rt, test_scores[:nH], test_target, ALPHA, config)
        rows.append({'heads': nH, 'coverage': float(cov), 'set_size': float(msz)})
        if nH == cal_output.shape[0]:
            last_bundle = (config, Dc, Dt, Rc, Rt, pred_sets)
    return pd.DataFrame(rows), last_bundle

def per_class_coverage_df(pred_sets, y_true):
    prediction_sets_all = pred_sets.all(axis=0)
    pred_set_list = [set(np.where(r)[0]) for r in prediction_sets_all]
    rows = []
    for c in range(NUM_CLASSES):
        idx = np.where(y_true == c)[0]
        coverage = float(np.mean([c in pred_set_list[j] for j in idx])) if idx.size > 0 else np.nan
        rows.append({'Class': f'Class {c}', 'Coverage': coverage})
    return pd.DataFrame(rows)

def build_binary_uncertainty_outputs(model, padded_x, y_raw, config, Dc, Dt, Rc, Rt):
    image_outputs = get_image_multi_head_outputs(model, padded_x, H, W, B, P_S, batch_size=BATCH_SIZE)
    image_scores = np.round(compute_scores(image_outputs, config), 4)
    y_flat = y_raw.ravel()
    orig_mask = np.zeros((H, W), dtype=bool)
    orig_mask[:330, :307] = True
    orig_mask_flat = orig_mask.ravel()
    gt_uncertain = (y_flat == 7) & orig_mask_flat
    cp_valid = orig_mask_flat & (~gt_uncertain)
    img_valid = image_scores[:, cp_valid, :]
    y_valid = y_flat[cp_valid] - 1
    cov, mset, pred_bool = main_algo(Dc, Dt, Rc, Rt, img_valid, y_valid, config['ALPHA'], config)
    set_sizes = pred_bool.sum(axis=2).mean(axis=0)
    u_valid = set_sizes / float(NUM_CLASSES)

    # --- THIS IS THE ONLY CHANGE: quantile threshold instead of top-K ---
    thresh = np.nanquantile(u_valid, 1 - UNCERTAIN_FRACTION)
    cp_uncertain_valid = u_valid >= thresh
    # --- END OF CHANGE ---

    cp_uncertain = np.zeros(H * W, dtype=bool)
    cp_valid_indices = np.where(cp_valid)[0]
    cp_uncertain[cp_valid_indices[cp_uncertain_valid]] = True
    final_uncertain = cp_uncertain | gt_uncertain
    avg_probs = np.mean(image_outputs, axis=0)
    class_pred = np.argmax(avg_probs, axis=1)
    class_map = np.full(H * W, np.nan)
    class_map[orig_mask_flat] = class_pred[orig_mask_flat]
    display_map = class_map.copy()
    display_map[final_uncertain] = np.nan
    binary_map = np.zeros(H * W, dtype=np.int32)
    binary_map[orig_mask_flat] = 0
    binary_map[final_uncertain] = 1
    visible_display = np.where(np.isnan(display_map.reshape(H, W)), -1, display_map.reshape(H, W))
    disp_real = visible_display[orig_mask]
    class_counts = [int(np.sum(disp_real == c)) for c in range(NUM_CLASSES)]
    uncertain_total = int(np.sum(binary_map.reshape(H, W)[orig_mask] == 1))
    return {
        'coverage': float(cov),
        'mean_set_size': float(mset),
        'binary_uncertainty_map': binary_map.reshape(H, W),
        'display_map': visible_display,
        'class_pixel_counts': class_counts + [uncertain_total],
    }

def head_sweep_figure(df, model_name, scoring_method):
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    sns.lineplot(data=df, x='heads', y='coverage', marker='o', ax=axes[0], color='#4C72B0')
    axes[0].axhline(1 - ALPHA, linestyle='--', color='red', linewidth=2)
    axes[0].set_title(f'{model_name} {scoring_method}: Coverage')
    sns.lineplot(data=df, x='heads', y='set_size', marker='o', ax=axes[1], color='#55A868')
    axes[1].set_title(f'{model_name} {scoring_method}: Set Size')
    plt.tight_layout(); plt.show(); return fig

def per_class_figure(df, model_name, scoring_method):
    fig, ax = plt.subplots(figsize=(15, 7))
    ax.bar(df['Class'], df['Coverage'], color='skyblue', edgecolor='black')
    ax.axhline(
        y=1 - ALPHA, color='red', linestyle='--', linewidth=2,
        label=f'Desired Coverage ({1 - ALPHA:.2f})',
    )
    ax.set_title(f'Per-Class Coverage ({scoring_method}) - {model_name}', fontsize=16)
    ax.set_xlabel('Class')
    ax.set_ylabel('Coverage')
    ax.set_ylim([0, 1.1])
    ax.grid(axis='y', linestyle='--', alpha=0.7)
    ax.tick_params(axis='x', rotation=45)
    add_bar_labels(ax, fmt='{:.2f}', y_pad=0.01)
    ax.legend(loc='upper right')
    plt.tight_layout()
    plt.show()
    return fig

def binary_uncertainty_figure(binary_map, model_name):
    disp = binary_map.astype(int)
    cmap = ListedColormap([CERTAIN_COLOR, UNCERTAIN_MAP_COLOR])
    fig, ax = plt.subplots(figsize=(10, 8))
    ax.imshow(disp, cmap=cmap, vmin=0, vmax=1)
    ax.set_title(
        f'Predictions with {int((1 - ALPHA) * 100)}% Uncertainty Map\n'
        f'(MultiCP — {model_name})',
        fontsize=16,
    )
    ax.axis('off')
    legend_handles = [
        Patch(facecolor=CERTAIN_COLOR, edgecolor='black', label='Certain'),
        Patch(facecolor=UNCERTAIN_MAP_COLOR, edgecolor='black', label='Uncertain'),
    ]
    ax.legend(
        handles=legend_handles,
        loc='upper left',
        bbox_to_anchor=(1.02, 1),
        borderaxespad=0.0,
        frameon=True,
    )
    plt.tight_layout()
    plt.show()
    return fig

def prediction_map_figure(display_map, model_name):
    combined_map = np.where(display_map == -1, NUM_CLASSES, display_map).astype(int)
    cmap = ListedColormap(CLASS_COLORS + [UNCERTAIN_COLOR])
    fig, ax = plt.subplots(figsize=(12, 10))
    im = ax.imshow(combined_map, cmap=cmap, vmin=0, vmax=NUM_CLASSES)
    ax.set_title(
        f'Predictions with {int((1 - ALPHA) * 100)}% Uncertainty Mask\n'
        f'(MultiCP — {model_name})',
        fontsize=16,
    )
    ax.axis('off')
    cbar = fig.colorbar(
        im, ax=ax,
        ticks=np.arange(NUM_CLASSES + 1),
        fraction=0.046,
        pad=0.04,
    )
    cbar.set_ticklabels(
        [f'Class {i}' for i in range(NUM_CLASSES)] + ['Uncertain']
    )
    plt.tight_layout()
    plt.show()
    return fig

def pixel_count_figure(class_pixel_counts, model_name):
    labels = [f'Class {i}' for i in range(NUM_CLASSES)] + ['Uncertain']
    colors = CLASS_COLORS + [UNCERTAIN_COLOR]
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.bar(labels, class_pixel_counts, color=colors[:len(labels)], edgecolor='black')
    ax.tick_params(axis='x', rotation=45)
    ax.set_ylabel('Number of Pixels')
    ax.set_title(
        f'Pixel Count per Class (Including Uncertain Regions) — {model_name}',
        fontsize=16,
    )
    ax.grid(axis='y', linestyle='--', alpha=0.5)
    ymax = max(class_pixel_counts) if len(class_pixel_counts) else 1
    for i, v in enumerate(class_pixel_counts):
        ax.text(i, v + 0.01 * ymax, f'{int(v):,}', ha='center', va='bottom', fontsize=9)
    plt.tight_layout()
    plt.show()
    return fig

def visualize_cell_selection(Dcells_scores, Dcells_target, D_i_order, model_name):
    pts = Dcells_scores[:, :2] if Dcells_scores.shape[1] > 2 else Dcells_scores
    vor = Voronoi(pts)
    ranks = np.argsort(D_i_order)
    normalized_order = np.zeros(len(pts))
    normalized_order[ranks] = np.linspace(0, 1, len(pts))
    fig, ax = plt.subplots(figsize=(7, 6))
    voronoi_plot_2d(vor, ax=ax, show_vertices=False, show_points=True, line_colors='black', line_width=0.3)
    for region_idx, region in enumerate(vor.point_region):
        polygon = vor.regions[region]
        if not -1 in polygon and len(polygon) > 0:
            poly_coords = [vor.vertices[i] for i in polygon]
            ax.fill(*zip(*poly_coords), color=cm.magma(1 - normalized_order[region_idx]), alpha=0.9)
    sc = ax.scatter(pts[:, 0], pts[:, 1], c=normalized_order, cmap='magma_r', edgecolor='white', s=40)
    plt.colorbar(sc, ax=ax, label='Selection Order (Dark=Earlier, Light=Later)')
    ax.set_title(f'Voronoi Cell Selection Order - {model_name}')
    plt.tight_layout(); plt.show()
    df = pd.DataFrame({'Cell Index': np.arange(len(pts)), 's1': pts[:, 0], 's2': pts[:, 1], 'Selection_Order': D_i_order, 'Norm_Order(0-1)': normalized_order, 'Target': Dcells_target})
    return fig, df



In [ ]:
registry = json.loads(MODEL_REGISTRY_PATH.read_text())
wb = ensure_workbook(WORKBOOK_PATH)
if 'Summary' in wb.sheetnames and len(wb.sheetnames) > 1:
    for sheet in wb.sheetnames[1:]:
        del wb[sheet]
summary_rows = []
for model_key, info in registry.items():
    model_name = MODEL_NAME_MAP.get(model_key, model_key)
    print('\n' + '=' * 25 + f' Loading {model_name} ' + '=' * 25)
    model = load_model(info['best_model_path'], custom_objects=CUSTOM_OBJECTS, compile=False, safe_mode=False)
    cal_output = get_multihead_outputs(model, x_cal, batch_size=BATCH_SIZE)
    test_output = get_multihead_outputs(model, x_test_new, batch_size=BATCH_SIZE)
    for scoring_method in SCORING_METHODS:
        head_df, bundle = compute_head_sweep(cal_output, test_output, y_cal.astype(np.int32), y_test_new.astype(np.int32), scoring_method)
        config, Dc, Dt, Rc, Rt, pred_sets = bundle
        class_cov_df = per_class_coverage_df(pred_sets, y_test_new.astype(np.int32))
        binary_outputs = build_binary_uncertainty_outputs(model, padded_x, y, config, Dc, Dt, Rc, Rt)
        sweep_fig = head_sweep_figure(head_df, model_name, scoring_method)
        class_fig = per_class_figure(class_cov_df, model_name, scoring_method)
        binary_fig = binary_uncertainty_figure(binary_outputs['binary_uncertainty_map'], model_name)
        pred_fig = prediction_map_figure(binary_outputs['display_map'], model_name)
        counts_fig = pixel_count_figure(binary_outputs['class_pixel_counts'], model_name)
        D_i_order = np.argsort(-np.mean(Dc, axis=1))
        vor_fig, vor_df = visualize_cell_selection(Dc, Dt, D_i_order, model_name)
        sheet_name = f"{model_key[:10]}_{scoring_method}"[:31]
        ws = wb.create_sheet(title=sheet_name)
        summary = {
            'model_key': model_key,
            'model_name': model_name,
            'scoring_method': scoring_method,
            'empirical_coverage': float(head_df.iloc[-1]['coverage']),
            'avg_set_size': float(head_df.iloc[-1]['set_size']),
            'scene_coverage': float(binary_outputs['coverage']),
            'scene_avg_set_size': float(binary_outputs['mean_set_size']),
            'uncertain_pixel_rate': float(binary_outputs['binary_uncertainty_map'].mean()),
            'model_path': info['best_model_path'],
        }
        for idx, (k, v) in enumerate(summary.items(), start=1):
            ws.cell(row=idx, column=1, value=k)
            ws.cell(row=idx, column=2, value=v)
        write_df(ws, head_df, start_row=12, start_col=1)
        write_df(ws, class_cov_df, start_row=12, start_col=6)
        write_df(ws, pd.DataFrame({'Class': [f'Class {i}' for i in range(NUM_CLASSES)] + ['Uncertain'], 'Pixels': binary_outputs['class_pixel_counts']}), start_row=12, start_col=10)
        write_df(ws, vor_df, start_row=40, start_col=1)
        for anchor, fig in [('N2', sweep_fig), ('N28', class_fig), ('N54', binary_fig), ('V54', pred_fig), ('N80', counts_fig), ('V2', vor_fig)]:
            add_image(ws, fig, anchor)
        autosize_columns(ws)
        summary_rows.append(summary)
summary_df = pd.DataFrame(summary_rows)
summary_ws = wb['Summary']
summary_ws.delete_rows(1, summary_ws.max_row)
write_df(summary_ws, summary_df, start_row=1, start_col=1)
autosize_columns(summary_ws)
wb.save(WORKBOOK_PATH)
print('Saved MultiCP workbook to', WORKBOOK_PATH)
summary_df



Output hidden; open in https://colab.research.google.com to view.